# Exercise - Assignment task-Find the best text classification model for the sentimental analysis (assignment submission)

The tasks for this part is use the grid search to:
1. Identify which vectorization method works the best or basically not much difference.
2. Identify which model, together with its corresponding hyperparameters, gives the best performance for traffic sentimental analysis.

You can either use the structure below or be a be a bit more explorative and try out other strategies we have discussed in the lecture/exercises to find the best parameters/model (e.g., Random Search, ROC curve,...).

In [29]:
from sklearn.base import ClassifierTags
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
%matplotlib inline
from sklearn.metrics import ConfusionMatrixDisplay as cmd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB
import os
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# The path of the dataset
url = 'https://raw.githubusercontent.com/zhenliangma/Applied-AI-in-Transportation/master/Exercise_4_Text_classification/Pakistani%20Traffic%20sentiment%20Analysis.csv'

# Load the data use the pandas
df = pd.read_csv(url)

#-*-*-*-*-*-*chose different vectorization-*-*-*-*-*-*

# Modification to run the 3 vectorization methods

#(1) CountVectorizer
# vectorizerCV = CountVectorizer(ngram_range=(1, 2), stop_words='english',min_df=20)
vectorizerCV = CountVectorizer(ngram_range=(1, 2), stop_words=None ,min_df=20)

#(2) #HashingVectorizer
vectorizerHV = HashingVectorizer(ngram_range=(1, 2), n_features=200)

#(3)TfidfVectorizer
vectorizerTV = TfidfVectorizer(
    min_df=20,
    norm='l2',
    smooth_idf=True,
    use_idf=True,
    ngram_range=(1, 1),
#    stop_words='english'
    stop_words=None
    )

# Creates a dictionary with the parameters for the 3 vectorization methods
vectorizer={'CountVectorizer':vectorizerCV,'HashingVectorizer':vectorizerHV,'TfidfVectorizer':vectorizerTV}

#-*-*-*-*-*-*chose different vectorization-*-*-*-*-*-*

# split into train/test set
x = df['Text']
y = df['Sentiment']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

# Creates a dictionary with the 3 vectorization transformed data
transformed_data = {}
# Creates an array to store the results and compare it
comparison_results = []

for name, vec in vectorizer.items():
    if name == 'HashingVectorizer':
        x_train_vectorized = vec.transform(x_train)
        x_test_vectorized = vec.transform(x_test)
    else:
        x_train_vectorized = vec.fit_transform(x_train)
        x_test_vectorized = vec.transform(x_test)

    print(f"\n\n  Method: {name} ready to train model.")

    # Define classification models to evaluate and its parameter grid for the optimization
    classifiers = {
        'LogisticRegression': (LogisticRegression(max_iter=1000, random_state=0),
                        {'C': [0.001, 0.01, 0.1, 1, 10, 100]}),
        'KNN': (KNeighborsClassifier(),{'n_neighbors': [3, 5, 7, 9],'weights': ['uniform', 'distance']}),
        # 'RandomForest': (RandomForestClassifier(random_state=0),{'n_estimators': [100, 200, 300],
        #                 'max_depth': [None, 10, 20, 30],
        #                 'min_samples_split': [2, 5, 10],
        #                 'min_samples_leaf': [1, 2, 4]}),
        'XGBoost': (XGBClassifier(),{'learning_rate': [0.01, 0.1, 0.2],
                    'n_estimators': [100, 200, 300],
                    'max_depth': [3, 4, 5]}),
        'SVM': (SVC(probability=True),{'kernel': ['linear', 'rbf', 'poly'],'C': [0.1, 1, 10]}),
        'Naïve Bayes': (BernoulliNB(),{'alpha': [0.1, 0.5, 1],'force_alpha': [True,False]})
         }

    # Bucle to execute each classification model with the GridSearch to find the best hyperparameters
    for modelname, (model, param_grid) in classifiers.items():
        print(f"\n--- Optimizing {modelname} ---")

        #`grid_search` performs a grid search with 5-fold cross-validation and evaluates models based on accuracy.
        grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy',n_jobs=-1)

        #`fit` method fits the model to the training data, systematically trying out all parameter combinations.
        grid_search.fit(x_train_vectorized, y_train)

        #`best_params` and `best_score` store the best hyperparameters and their corresponding accuracy score.
        best_params = grid_search.best_params_
        print(best_params)
        best_score = grid_search.best_score_

        # recover the trained and optimized model
        best_model = grid_search.best_estimator_

        # Predict test samples
        y_pred = best_model.predict(x_test_vectorized)

        # Predict test samples and compute accuracy
        #y_pred = model.predict(x_test_vectorized)
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Record the performance metrics
        comparison_results.append({
            'Model': modelname,
            'Vectorizer': name,
            'Accuracy Score': round(acc, 4),
            'Precision Score': round(prec, 4),
            'Recall Score': round(rec, 4),
            'F1 Score': round(f1, 4)
        })

# Convert summary list to a structured DataFrame and sort by performance
df_comparison = pd.DataFrame(comparison_results).sort_values(by='Accuracy Score', ascending=False)

# Display the final comparative table
print(df_comparison.to_string(index=False))

# cmd.from_estimator(
#     best_model,
#     x_test_vectorized,
#     y_test,
#     display_labels=['Negative', 'Positive'],
#     cmap='Blues',
#     xticks_rotation='vertical'
# )


Method: CountVectorizer ready to train model.

--- Optimizing LogisticRegression ---
{'C': 1}

--- Optimizing KNN ---
{'n_neighbors': 9, 'weights': 'distance'}

--- Optimizing XGBoost ---
{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}

--- Optimizing SVM ---
{'C': 1, 'kernel': 'rbf'}

--- Optimizing Naïve Bayes ---
{'alpha': 1, 'force_alpha': True}
Method: HashingVectorizer ready to train model.

--- Optimizing LogisticRegression ---
{'C': 10}

--- Optimizing KNN ---
{'n_neighbors': 9, 'weights': 'distance'}

--- Optimizing XGBoost ---
{'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100}

--- Optimizing SVM ---
{'C': 1, 'kernel': 'rbf'}

--- Optimizing Naïve Bayes ---
{'alpha': 0.5, 'force_alpha': True}
Method: TfidfVectorizer ready to train model.

--- Optimizing LogisticRegression ---
{'C': 1}

--- Optimizing KNN ---
{'n_neighbors': 9, 'weights': 'distance'}

--- Optimizing XGBoost ---
{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}

--- Optimizing SVM

Results for First configuration:

Method: CountVectorizer ready to train model.

--- Optimizing LogisticRegression ---
{'C': 10}

--- Optimizing KNN ---
{'n_neighbors': 9, 'weights': 'distance'}

--- Optimizing RandomForest ---
{'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 100}

--- Optimizing XGBoost ---
{'learning_rate': 0.2, 'max_depth': 4, 'n_estimators': 200}

--- Optimizing SVM ---
{'C': 1, 'kernel': 'rbf'}

--- Optimizing Naïve Bayes ---
{'alpha': 0.5, 'force_alpha': True}
Method: HashingVectorizer ready to train model.

--- Optimizing LogisticRegression ---
{'C': 10}

--- Optimizing KNN ---
{'n_neighbors': 9, 'weights': 'distance'}

--- Optimizing RandomForest ---
{'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 300}

--- Optimizing XGBoost ---
{'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100}

--- Optimizing SVM ---
{'C': 1, 'kernel': 'rbf'}

--- Optimizing Naïve Bayes ---
{'alpha': 0.5, 'force_alpha': True}
Method: TfidfVectorizer ready to train model.

--- Optimizing LogisticRegression ---
{'C': 1}

--- Optimizing KNN ---
{'n_neighbors': 9, 'weights': 'distance'}

--- Optimizing RandomForest ---
{'max_depth': 30, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 300}

--- Optimizing XGBoost ---
{'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 300}

--- Optimizing SVM ---
{'C': 1, 'kernel': 'rbf'}

--- Optimizing Naïve Bayes ---
{'alpha': 1, 'force_alpha': True}
             Model        Vectorizer  Accuracy Score  Precision Score  Recall Score  F1 Score
               SVM HashingVectorizer          0.9763           0.9694        0.9867    0.9780
      RandomForest HashingVectorizer          0.9763           0.9736        0.9822    0.9779
      RandomForest   CountVectorizer          0.9716           0.9733        0.9733    0.9733
      RandomForest   TfidfVectorizer          0.9692           0.9649        0.9778    0.9713
           XGBoost   TfidfVectorizer          0.9692           0.9732        0.9689    0.9710
           XGBoost HashingVectorizer          0.9645           0.9646        0.9689    0.9667
LogisticRegression   TfidfVectorizer          0.9621           0.9644        0.9644    0.9644
               SVM   TfidfVectorizer          0.9621           0.9563        0.9733    0.9648
           XGBoost   CountVectorizer          0.9597           0.9771        0.9467    0.9616
               SVM   CountVectorizer          0.9597           0.9815        0.9422    0.9615
       Naïve Bayes   TfidfVectorizer          0.9526           0.9724        0.9378    0.9548
               KNN   CountVectorizer          0.9502           0.9766        0.9289    0.9522
LogisticRegression HashingVectorizer          0.9502           0.9513        0.9556    0.9534
       Naïve Bayes   CountVectorizer          0.9502           0.9679        0.9378    0.9526
LogisticRegression   CountVectorizer          0.9479           0.9765        0.9244    0.9498
               KNN HashingVectorizer          0.9431           0.9548        0.9378    0.9462
               KNN   TfidfVectorizer          0.9147           0.9091        0.9333    0.9211
       Naïve Bayes HashingVectorizer          0.8175           0.8524        0.7956    0.8230

In [28]:
# Here you change the reviews
text = 'Adayala road is not clear'

# Vectorize with HashingVectorizer from dicciionary
text_vec = vectorizerHV.transform([text])

# 3. Train the best configuration from GridSearch ({'C': 1, 'kernel': 'rbf'})
from sklearn.svm import SVC
winner_model = SVC(C=1, kernel='rbf', probability=True, random_state=0)
winner_model.fit(vectorizerHV.transform(x_train), y_train)

# Make a prediction for this review
# score=model.predict_proba(vectorizer.transform([text]))[0][1]
score = winner_model.predict_proba(text_vec)[0][1]

if score >0.5:
  attitude='negative'
else:
  attitude='positive'

print('The prediction result of this review is: '+ attitude)

The prediction result of this review is: positive
